# 04 - Entrenamiento secuencial B2 → B3 → B4

Este cuaderno ejecuta en orden la comprobación de las finalistas de B2 (`C=1.0` y `C=0.1` en A, B y C), la búsqueda completa de B3 y la búsqueda completa de B4. Al terminar una fase comienza automáticamente la siguiente. Usa todas las partidas de `train` en A, B y C; `val` solo compara finalistas y `test` no se toca.

Los estudios Optuna, resultados y checkpoints se guardan en Google Drive. En B2, cada combinación pliegue/`C` guarda sus métricas al terminar, antes de entrenar el siguiente `C`; si Colab se interrumpe, se omiten las combinaciones ya guardadas y se continúa con la primera pendiente. `pipeline_status.json` registra cada modelo/escenario: las fases completas se saltan enteras al reanudar y B3/B4 continúan desde sus checkpoints de trial. Un trial interrumpido se repite desde el comienzo.


In [ ]:
from pathlib import Path
import sys
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
# Carga el código actualizado después de instalar dependencias.
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa la descarga anterior.'
from ptcg_wp.pipeline import run_sequential_pipeline
from sklearn import __version__ as sklearn_version
assert tuple(map(int, sklearn_version.split('.')[:2])) >= (1, 7), 'B4 necesita scikit-learn >=1.7; reinicia el runtime tras instalar.'
print('Entorno listo | sklearn:', sklearn_version, '| Drive:', CHECKPOINT_ROOT)

In [ ]:
# Corrida completa y secuencial; cada fase conserva sus propios checkpoints.
pipeline_outputs = run_sequential_pipeline(
    data_dir=DATA_DIR,
    checkpoint_root=CHECKPOINT_ROOT,
    batch_size=32768,
    b2_max_epochs=600,  # Doble del presupuesto de 300 de la primera búsqueda.
    b2_eta0=0.05,
    b2_gradient_tol=1e-3,
    b3_trials=10,
    b3_n_jobs=2,
    b4_trials=10,
    b4_max_iter=500,
    b4_patience=30,
    max_ram_gib=10.0,
    seed=2026,
    verbose=True,
)
print('Resultados/checkpoints por fase:')
for fase, ruta in pipeline_outputs.items():
    print(f'{fase}: {ruta}')